# ARC-AGI-3 â€” MuZero Pre-training (Stage 1)

Three-stage training pipeline:
- **Stage 1 (this notebook)**: Broad pre-training on Atari-HEAD + ARC-AGI-3 domain anchors
- **Stage 2**: Domain-adaptive fine-tuning on augmented 4-game recordings
- **Stage 3**: TTT at deployment (two scopes: intra-level + cross-level)

**Hardware**: Kaggle T4Ã—2 (32GB VRAM total)

**Data mix**: ~98% Atari-HEAD + ~2% ARC-AGI-3 recordings, ARC loss upweighted 3Ã—

**Key constraint**: NO color jitter on ARC-AGI-3 data â€” colors are semantic identifiers

In [ ]:
import os, sys, subprocess, zipfile, pathlib

# ── Paths ────────────────────────────────────────────────────────────────────
KAGGLE_INPUT   = '/kaggle/input'
KAGGLE_WORK    = '/kaggle/working'
ARC3_REC_DIR   = f'{KAGGLE_INPUT}/arc3-recordings'  # Kaggle dataset slug
MUZERO_DIR     = f'{KAGGLE_WORK}/muzero-general'
ARC3_DIR       = f'{KAGGLE_WORK}/arc3'
ATARI_HEAD_WORK = f'{KAGGLE_WORK}/atari-head'  # extracted game dirs live here

# ── Clone repos ──────────────────────────────────────────────────────────────
if not os.path.exists(MUZERO_DIR):
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/werner-duvaud/muzero-general.git',
                    MUZERO_DIR], check=True)

if not os.path.exists(ARC3_DIR):
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/humanaiconvention/arc3.git',
                    ARC3_DIR], check=True)

# Add to path
for p in [MUZERO_DIR, f'{ARC3_DIR}/neurosym']:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Extract Atari-HEAD game zips ─────────────────────────────────────────────
# Kaggle datasets are NOT auto-extracted. We unzip each game zip into
# KAGGLE_WORK/atari-head/<game>/ so AtariHeadDataset can find the .txt + .tar.bz2.
os.makedirs(ATARI_HEAD_WORK, exist_ok=True)
_atari_input = pathlib.Path(f'{KAGGLE_INPUT}/atari-head')
_zip_files   = sorted(_atari_input.glob('*.zip'))
print(f'Extracting {len(_zip_files)} game zips to {ATARI_HEAD_WORK} ...')
for _zp in _zip_files:
    _game = _zp.stem  # e.g. "breakout"
    _dest = pathlib.Path(ATARI_HEAD_WORK) / _game
    if _dest.exists():
        print(f'  {_game}: already extracted, skipping')
        continue
    print(f'  {_game}: extracting {_zp.stat().st_size // 1_048_576} MB ...')
    with zipfile.ZipFile(_zp) as _zf:
        _zf.extractall(ATARI_HEAD_WORK)
    print(f'  {_game}: done')

# AtariHeadDataset reads from here
ATARI_HEAD_DIR = ATARI_HEAD_WORK
print('
Setup complete.')
print(f'  MUZERO_DIR     : {MUZERO_DIR}')
print(f'  ARC3_DIR       : {ARC3_DIR}')
print(f'  ATARI_HEAD_DIR : {ATARI_HEAD_DIR}')
print(f'  ARC3_REC_DIR   : {ARC3_REC_DIR}')


In [ ]:
# â”€â”€ Install deps â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
subprocess.run([
    'pip', 'install', '-q',
    'torch', 'torchvision',
    'ray[default]',
    'tensorboard',
    'Pillow',
    'numpy',
], check=True)
print('Deps installed.')

In [ ]:
import numpy as np
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {name} ({mem:.1f} GB)')

## 1. Load data

In [ ]:
from data_pipeline import ARC3Dataset, AtariHeadDataset, apply_domain_upweight

# ARC-AGI-3 domain anchors
arc3_ds = ARC3Dataset(ARC3_REC_DIR)
arc3_histories = arc3_ds.load_all(verbose=True)

# Atari-HEAD
atari_ds = AtariHeadDataset(ATARI_HEAD_DIR, max_trials_per_game=2, max_steps_per_trial=500)
atari_histories = atari_ds.load_all(verbose=True)

print(f'\nRaw counts: {len(atari_histories)} Atari, {len(arc3_histories)} ARC3')

In [ ]:
# Mix: ARC3 domain anchors at 3Ã— resample + will receive 3Ã— loss weight
all_histories = atari_histories + apply_domain_upweight(arc3_histories, upweight=3.0)

import random
random.shuffle(all_histories)

n_anchor = sum(1 for h in all_histories if h.is_domain_anchor)
pct = 100 * n_anchor / len(all_histories)
print(f'Mixed corpus: {len(all_histories)} histories')
print(f'Domain anchor: {n_anchor} ({pct:.1f}%) â€” target 1.5â€“2.5%')
print(f'Approx total steps: {sum(len(h.observation_history) for h in all_histories):,}')

## 2. MuZero config

In [ ]:
import pathlib, datetime
import sys
sys.path.insert(0, MUZERO_DIR)
sys.path.insert(0, f'{ARC3_DIR}/neurosym')

from arc3_game import MuZeroConfig

config = MuZeroConfig(game_id='pretrain', n_bins=4)

# Override for pre-training on mixed corpus
# Observation shape is fixed: (1, 64, 64)
# Action space: use largest seen (18 for Atari; ARC3 has max 16)
config.action_space        = list(range(18))
config.observation_shape   = (1, 64, 64)
config.stacked_observations = 0

# Training budget â€” Stage 1
config.training_steps      = 50_000
config.batch_size          = 128
config.replay_buffer_size  = 5_000
config.num_unroll_steps    = 5
config.td_steps            = 20

# LR schedule
config.lr_init             = 3e-4
config.lr_decay_rate       = 0.9
config.lr_decay_steps      = 10_000

# Network â€” small ResNet for fast TTT
config.network             = 'resnet'
config.downsample          = 'CNN'
config.blocks              = 3
config.channels            = 64

# Output path
config.results_path = pathlib.Path(KAGGLE_WORK) / 'results' / 'pretrain' / \
    datetime.datetime.now().strftime('%Y-%m-%d--%H-%M-%S')
config.results_path.mkdir(parents=True, exist_ok=True)

config.train_on_gpu        = torch.cuda.is_available()
config.num_workers         = 2   # one per T4

print('Config ready.')
print(f'  action_space size : {len(config.action_space)}')
print(f'  observation_shape : {config.observation_shape}')
print(f'  training_steps    : {config.training_steps:,}')
print(f'  train_on_gpu      : {config.train_on_gpu}')
print(f'  results_path      : {config.results_path}')

## 3. Domain-upweight loss wrapper

ARC-AGI-3 domain-anchor samples receive 3Ã— loss weight.
This is applied at the batch level â€” no distribution shift from resampling.

In [ ]:
# Patch muzero-general's trainer to support per-sample loss weights
# The GameHistory.is_domain_anchor flag drives the weight.

import importlib
import trainer as _trainer_module

DOMAIN_ANCHOR_LOSS_WEIGHT = 3.0

_orig_update_weights = _trainer_module.Trainer.update_weights

def _patched_update_weights(self, batch):
    """
    Wraps Trainer.update_weights to apply 3Ã— loss scaling on
    domain-anchor samples (ARC-AGI-3 recordings).
    
    The `batch` tuple from the replay buffer includes a `is_domain_anchor`
    flag when present; otherwise defaults to 1.0 weight.
    """
    # Standard update â€” domain weighting applied inside if flag present
    return _orig_update_weights(self, batch)

# NOTE: Full loss-weight injection requires patching the loss computation
# inside update_weights. The above stub marks the integration point.
# For the first training run, the 3Ã— resampling in apply_domain_upweight
# provides the approximate upweighting without code surgery.
print('Loss upweight: using 3Ã— resampling as proxy for loss scaling (Stage 1).')
print('Loss-weight injection into trainer.update_weights is TODO for Stage 2.')

## 4. Seed replay buffer with offline data

In [ ]:
import ray
import copy

# Convert our GameHistory dataclass to muzero-general's GameHistory class
import self_play as _self_play_module
MZGameHistory = _self_play_module.GameHistory

def to_mz_history(h) -> MZGameHistory:
    """Convert data_pipeline.GameHistory â†’ muzero-general GameHistory."""
    mz = MZGameHistory()
    mz.observation_history = h.observation_history
    mz.action_history      = h.action_history
    mz.reward_history      = h.reward_history
    mz.to_play_history     = h.to_play_history
    mz.child_visits        = h.child_visits
    mz.root_values         = h.root_values
    return mz

if not ray.is_initialized():
    ray.init(num_gpus=torch.cuda.device_count(), ignore_reinit_error=True)

# Build initial checkpoint and buffer
checkpoint = {
    'weights'         : None,
    'optimizer_state' : None,
    'total_reward'    : 0,
    'muzero_reward'   : 0,
    'opponent_reward' : 0,
    'episode_length'  : 0,
    'mean_value'      : 0,
    'training_step'   : 0,
    'lr'              : 0,
    'total_loss'      : 0,
    'value_loss'      : 0,
    'reward_loss'     : 0,
    'policy_loss'     : 0,
    'num_played_games': 0,
    'num_played_steps': 0,
    'num_reanalysed_games': 0,
    'terminate'       : False,
}

# Seed buffer with offline histories (up to replay_buffer_size)
seed_histories = all_histories[:config.replay_buffer_size]
initial_buffer = {i: to_mz_history(h) for i, h in enumerate(seed_histories)}
checkpoint['num_played_games'] = len(initial_buffer)
checkpoint['num_played_steps'] = sum(len(h.action_history) for h in seed_histories)

print(f'Seeded replay buffer with {len(initial_buffer)} histories')
print(f'Total seed steps: {checkpoint["num_played_steps"]:,}')

## 5. Train

In [ ]:
import models
import trainer as _trainer_module
import shared_storage
import replay_buffer as _replay_module

# Initialise model
muzero_model = models.MuZeroNetwork(config)
checkpoint['weights'] = copy.deepcopy(muzero_model.get_weights())

# Ray actors
storage = shared_storage.SharedStorage.remote(checkpoint, config)
replay  = _replay_module.ReplayBuffer.remote(checkpoint, initial_buffer, config)

# Trainer
tr = _trainer_module.Trainer.remote(checkpoint, config)

print(f'Starting Stage 1 pre-training: {config.training_steps:,} steps')
print(f'Model params: {sum(p.numel() for p in muzero_model.parameters()):,}')

# Training loop â€” runs until training_steps reached or manually interrupted
import time
t0 = time.time()

for i in range(0, config.training_steps, config.checkpoint_interval):
    tr.run_selfplay.remote(checkpoint, replay, storage)  # no-op: offline pre-training
    tr.update_weights.remote(replay, storage)            # gradient steps

    if i % 500 == 0:
        info = ray.get(storage.get_infos.remote())
        elapsed = (time.time() - t0) / 60
        print(f'  step {i:6,} | loss={info.get("total_loss", 0):.4f} '
              f'| elapsed={elapsed:.1f}m')

print('Stage 1 pre-training complete.')

## 6. Save checkpoint

In [ ]:
import pickle

final_weights = ray.get(storage.get_weights.remote())
ckpt_path = config.results_path / 'stage1_pretrain.checkpoint'

with open(ckpt_path, 'wb') as f:
    pickle.dump({
        'weights'       : final_weights,
        'config'        : config,
        'training_steps': config.training_steps,
        'stage'         : 1,
    }, f)

print(f'Checkpoint saved: {ckpt_path}')
print(f'Copy to output: cp {ckpt_path} /kaggle/working/stage1_pretrain.checkpoint')
import shutil
shutil.copy(ckpt_path, f'{KAGGLE_WORK}/stage1_pretrain.checkpoint')

## Next: Stage 2 (domain-adaptive fine-tuning)

Load `stage1_pretrain.checkpoint`, then fine-tune exclusively on the
augmented ARC-AGI-3 recordings:
- LR = 1/10th of Stage 1 LR (3e-5)
- 20â€“50Ã— augmentation: spatial crops, mirroring, sub-episode windowing
- NO color jitter (ARC colors = object identity)
- Early stopping against held-out episode split

Stage 2 notebook: `arc3_muzero_finetune.ipynb`